# Day 14 — Weekly Review: Week 2, Python for Production (7 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: look at Week 2 honestly against what the repo shows, decide what to finish and what to postpone, and set up Week 3 (SQL and Postgres) so it starts on solid ground.

- Deliverable: day14/day14_notes.ipynb with your checklist and recap answers filled in; Day 4 finished and pushed; the Day 5 to 13 folders in your repo; a README pass; the coding mock done in the project chat.

- Where this sits: end of Week 2, Phase 1 (Foundation). Weeks 3 and 4 (SQL, then FastAPI) close Phase 1 with a REST API backed by a database.

Time split (5h): a review day is still a full working day. Most of it goes into the oldest open item, because a review that only re-reads notes changes nothing in the repo.

| Block | Time | What you do |
|---|---|---|
| Review | 1h | Recap, Week 2 table, checklist, mixed review, scores and plan |
| Catch-up | 2.5h | Bring the Day 5 to 13 folders into the repo, then finish Day 4 and push |
| DSA | 0.5h | Day 4's top words and balanced brackets |
| Career | 1h | Coding mock in chat (40 min), README pass (20 min) |

Open items first: on GitHub, main still has a single commit from 28 Oct: Days 1 to 3 finished, and Day 4 as starter files with all eight functions still raising NotImplementedError. Nothing from Days 5 to 13 is in the repo. Part of that is not on you: the scheduled task that writes these sessions cannot push to your repo, so the Day 5 to 13 starter folders reached you only as patch and zip files in the project chat. The Day 4 work is yours, though, and it has been open for ten days.

## Recap (2 minutes, from memory)

- What does the code below print, and about how many seconds does it take? Why is it not 4 seconds, and not 2?

- Your exporter calls a customer's CMS from 20 tasks at once. Why does it need an asyncio.Semaphore, and what does the customer see without one?

- A teammate calls requests.get() inside an async def. What goes wrong, and what are the two fixes?

Predict first, then run it:

In [ ]:
# Recap: predict what prints and roughly how many seconds it takes.
import asyncio
import time


async def fetch(n: int) -> int:
    await asyncio.sleep(1)  # stands in for a 1-second API call
    return n * 10


async def main() -> None:
    start = time.perf_counter()
    a = await fetch(1)  # await one at a time...
    b = await fetch(2)
    both = await asyncio.gather(fetch(3), fetch(4))  # ...then two together
    print(a, b, both, round(time.perf_counter() - start))


asyncio.run(main())

Answers are on the last page of today's session.

## Week 2 in one table

Six days, six habits. Say the right-hand column out loud without looking. Any row you cannot explain to a colleague in a minute goes on your revision list.

| Day | Big idea | The line to remember |
|---|---|---|
| 8 | Git: snapshots, branches, pull requests, rebase | Commit small with a message that says why; never commit a secret, and rotate one that leaks |
| 9 | The shell: paths, pipes, grep, environment variables, exit codes, ports | Exit code 0 means success; stdout is data, stderr is for people |
| 10 | pytest: asserts, fixtures, parametrize, fakes | Test the rules, not the lines; fake the network so tests run offline |
| 11 | Iterators, generators, decorators, context managers | Generators are lazy and one-shot; with guarantees the cleanup |
| 12 | Packages, pyproject.toml, uv, logging, ruff, mypy | One pyproject.toml plus a lock file; logs go to stderr; types are checked before run time |
| 13 | asyncio, async/await, httpx, Semaphore | Concurrency helps only while waiting; always cap the requests in flight |

The thread through the week: Week 1 taught you to write code that works on your laptop. Week 2 taught what makes it safe to hand to a customer: history and review (Git), automation (the shell and exit codes), proof (tests), repeatable installs (packaging) and speed without harm (async with limits). In an FDE interview, this is what separates "can code" from "can ship".

## Self-assessment checklist

Tick only what you can do without notes or search. Copy the list into the notebook and mark each line yes or no. A no is useful: it tells us exactly what to revise.

- Create a branch, make two commits with clear messages, push it and open a pull request with gh pr create.

- Explain the difference between merge and rebase, and get a lost commit back with git reflog.

- Count the errors in a log with a pipe of grep, sort and uniq -c, and say what exit code grep returns when it finds nothing.

- Read an environment variable in Python with os.environ.get() and explain why secrets live there.

- Write a pytest test with a fixture, a parametrized table of cases, and a check that an error is raised.

- Fake an HTTP call with monkeypatch so a test runs with no network.

- Write a generator that streams a large file line by line, and say why its memory use stays flat.

- Write a decorator with functools.wraps, and a context manager with @contextmanager.

- Set up a project with uv init, add a dependency, and explain what uv.lock is for.

- Configure logging once at the entry point, and log from a module with logging.getLogger(__name__).

- Run three HTTP calls at once with asyncio.gather or a TaskGroup, with at most two in flight.

## Check your understanding (predict first, then run)

Six questions from across the week, mixed up on purpose. Write every prediction in the notebook before you run anything. If one surprises you, find the line that caused it and explain it in one sentence. Questions 1 to 4 are one script:

In [ ]:
# Week 2 mixed review: write your prediction for each part, then run.
import asyncio
import functools
from contextlib import contextmanager


# 1 (Day 11) generators are lazy and one-shot
def numbered(lines):
    for n, line in enumerate(lines, start=1):
        print("reading", n)
        yield f"{n}: {line}"


gen = numbered(["a", "b"])
print("made gen")
print(list(gen))
print(list(gen))  # what is left in gen now?


# 2 (Day 11) a decorator, with and without functools.wraps
def shout(func):
    @functools.wraps(func)  # copies __name__ and the docstring over
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper


@shout
def greet(name: str) -> str:
    """Say hello."""
    return f"hello {name}"


print(greet("rohit"), greet.__name__)


# 3 (Day 11) context managers: what order, when an error happens?
@contextmanager
def step(name):
    print("enter", name)
    try:
        yield name
    finally:  # runs on success AND on error
        print("exit", name)


try:
    with step("outer"):
        with step("inner"):
            raise ValueError("boom")
except ValueError as e:
    print("caught", e)


# 4 (Day 13) gather: finish order vs result order
async def job(name, delay):
    await asyncio.sleep(delay)
    print("done", name)
    return name


async def main():
    return await asyncio.gather(job("slow", 0.2), job("fast", 0.1))


print(asyncio.run(main()))

Question 5 is about pytest. Save the file below as test_slug.py and predict the last line before you run python -m pytest -q test_slug.py:

In [ ]:
# Question 5, test_slug.py: predict pytest's summary line, then run it.
import pytest


def slugify(title: str) -> str:
    return title.strip().lower().replace(" ", "-")


@pytest.mark.parametrize(
    ("title", "expected"),
    [
        ("Hello World", "hello-world"),
        ("  FAQ ", "faq"),
        ("Q&A Page", "qa-page"),
    ],
)
def test_slugify(title, expected):
    assert slugify(title) == expected

Question 6 is about the shell. Run it in your terminal from any folder:

```bash
# Question 6: predict all three output lines, then run each line.
printf 'INFO ok\nERROR db\nERROR db\nWARN slow\n' > mini.log
grep ERROR mini.log | sort | uniq -c
grep -q TIMEOUT mini.log && echo found || echo "not found"
echo $?  # the exit code of the LAST command that ran
```

## Progress scores

Scores are 1 to 10 against the bar for a hireable FDE, where 7 means you could do it in an interview without help. They come from what the repo and your session answers show, not from what was taught. Two weeks of topics have been taught and the repo shows the first three days, so no score has moved since Day 7. That is a reading of the evidence, not a judgement of your ability.

| Area | Day 7 to Day 14 | Evidence (repo and sessions) | Next step |
|---|---|---|---|
| Python | 3 to 3 | Days 1 to 3 pushed. Day 4 starters untouched; Days 5 to 13 not in the repo | Day 4 today, then the catch-up lane all week |
| TypeScript / Node | 6 to 6 | 7 years of React; not tested this week | Node services in Phase 3 |
| DSA | 2 to 2 | 4 easy problems solved; about 20 open from Days 4 to 13 | Day 4's two today; 30 minutes daily |
| Backend & APIs | 1 to 1 | HTTP, requests and httpx taught on Days 6 and 13, not yet practised in the repo | Postgres this week, FastAPI in Week 4 |
| LLM application engineering | 1 to 1 | Not started | Claude API in Week 5 |
| RAG | 1 to 1 | Embedding theory from the IIIT-B programme; no hands-on yet | Weeks 6 and 7, Project 1 |
| Agents & MCP | 1 to 1 | Not started | Weeks 9 and 10 |
| Evals & reliability | 1 to 1 | pytest taught on Day 10; no tests in the repo yet | Day 10 in the catch-up lane, evals in Week 8 |
| Cloud & deployment | 1 to 1 | Git, shell and packaging taught; the repo has one commit | Docker and cloud in Weeks 10 and 11 |
| Enterprise integration | 5 to 5 | 7 years of CMS integrations; not yet shown in Python code | Day 6's export into Postgres on Day 18, then Project 2 |
| Data skills | 2 to 2 | JSON and CSV taught; pandas not started | SQL all week, pandas on Day 20 |
| Customer-facing skills | 5 to 5 | 14 years of delivery; provisional until a mock is run | Today's coding mock; a customer case mock in Week 4 |
| System design for AI | 1 to 1 | Not started | Week 13 |
| Job hunt & positioning | 2 to 2 | Public repo; the Day 7 LinkedIn post not confirmed yet | README pass today, the post this week |

If you have done some of this work on your laptop without pushing it, push it today and say so in the Day 14 review; the scores go up the same day. Work that lives only on your laptop is invisible to a hiring manager too.

## Plan adjustment

Day 7 set a rule: if the gap between what was taught and what is in the repo is still above two days on Day 14, Python stops blocking the plan and we move on to SQL. The gap is now ten days, so the rule applies. Moving on with no other change would fail, though: Day 18 loads Day 6's CMS export into Postgres, and Week 4's FastAPI app is tested with Day 10's pytest skills. So Week 3 keeps SQL as the main topic and adds a fixed catch-up lane.

- New daily split for Week 3: 1.5h theory · 2h SQL hands-on · 1h catch-up · 0.5h DSA. Revision moves into the catch-up hour: finishing old tasks is the best revision there is.

- Must finish, because later weeks build on them: Day 4 (files, JSON, errors), Day 5 (dataclasses), Day 6 (the CMS export to CSV) and Day 10 (tests).

- Do lightly: Day 8 (one real pull request is enough) and Day 13 (re-read and run the examples; the full task can wait for FastAPI in Week 4).

- Read only, task later: Day 9 (do drills.sh only), Day 11 and Day 12. Their ideas return in Weeks 4 and 10, and you will apply them there on project code.

- Push every day, even unfinished work, with a message like "Day 15: WIP, schema done, queries pending".

The catch-up lane, one item at a time:

| Catch-up item | When | Done when |
|---|---|---|
| Day 4: contact_store.py, main.py | Today | Both runs of main.py match the expected output in its docstring |
| Day 5: support_ticket.py | Day 15 | The SLA breach report prints for tickets_export.json |
| Day 6: cms_export.py | Days 16 and 17 | The CSV is written from mock_cms.py, with retries and a timeout |
| Day 8: one pull request | Day 17 | Days 5 and 6 merged into main through a pull request |
| Day 10: tests | Days 18 and 19 | test_contact_store.py and test_cms_export.py are green |
| Day 13: re-read and run | Day 20 | Your check-your-understanding answers are right; the task is optional |

## Honest calibration: the ₹1 Cr target

The calibration from Day 1 holds. A package near ₹1 Cr in India is a senior offer at a top AI company or one of a few well-funded AI startups, and those loops expect shipped LLM systems, strong system design and solid coding rounds. Sixteen weeks from zero Python does not reach that bar. Sixteen weeks can reach a strong first FDE, AI engineer or solutions engineer role, with three portfolio projects and your 14 years of delivery and CMS work behind them. From there, ₹1 Cr is a 12 to 24 month goal.

This week adds a harder fact. At the pace the repo shows, three days of work in seventeen calendar days, the 16-week plan would take well over a year. Applications are due to start in Week 8, around 13 Dec, and they need Project 1, a working RAG app. That date is still reachable, but only if the catch-up lane is done this week and every day from now on ends with a push. If 5 hours a day is not realistic right now, tell me the real number in the Day 14 review. A plan built on 3 honest hours beats a 5-hour plan that only exists on paper.

## Week 3 plan: SQL and Postgres

Almost every customer system you will connect to keeps its data in a relational database, and most AI features read from one: the CRM, the order system, the analytics warehouse, and later your own app's users and documents. Week 3 teaches SQL from zero on PostgreSQL, the database most AI startups run on, and ends with your CMS export living in Postgres. Install PostgreSQL 18 or newer before Day 15; on your Mac, Postgres.app is the simplest route and Homebrew works too. From Python you will use psycopg 3.

Daily split: 1.5h theory · 2h SQL hands-on · 1h catch-up · 0.5h DSA.

| Day | Date | Topic | Deliverable |
|---|---|---|---|
| 15 | 8 Nov | Relational thinking and first queries: tables, rows, keys, data types, psql, CREATE TABLE, INSERT, SELECT, WHERE, ORDER BY, LIMIT | day15/schema.sql and queries.sql: a CMS content model with 10 questions answered in SQL |
| 16 | 9 Nov | Aggregates and NULL: COUNT, SUM, AVG, GROUP BY, HAVING, DISTINCT, NULL and COALESCE, CASE | day16/report.sql: a content-ops report (entries per type, locale and status; stale drafts) |
| 17 | 10 Nov | Joins and modelling: primary and foreign keys, one-to-many, many-to-many, INNER and LEFT JOIN, normalisation | day17: a normalised schema for entries, authors, tags and locales, with join queries |
| 18 | 11 Nov | Python and Postgres: psycopg 3, parameterised queries and SQL injection, transactions | day18/load_export.py: loads Day 6's CMS export CSV into Postgres in one transaction |
| 19 | 12 Nov | Constraints, indexes and upserts: NOT NULL, UNIQUE, CHECK, ON CONFLICT, indexes, EXPLAIN ANALYZE, migrations | The loader made safe to re-run (upsert), and an index that speeds up a slow query, with EXPLAIN before and after |
| 20 | 13 Nov | Analytics SQL: CTEs, window functions, JSONB columns, pandas read_sql | day20 queries: latest version per entry, rank by views, search inside JSONB; a pandas summary |
| 21 | 14 Nov | Weekly review + scores + plan for Week 4 (FastAPI) | Review report, mock interview, catch-up lane closed |

Week 4 is FastAPI: a REST API over this database, with tests, which closes Phase 1. This table is also added to the plan at the top of the doc.

## Mock interview: coding

Type Mock interview: coding in the Forward Deployment Engineer project chat. I will play an engineer at an AI company running a 40-minute screen: one easy problem and one easy-to-medium problem on lists, dicts and strings, in plain Python, then a hire or no-hire verdict with specific feedback. This is what gets scored, in order:

- Clarify before you code: restate the problem, ask about the input (an empty list? duplicates? upper and lower case?) and agree on one example.

- Talk while you code. Five minutes of silence reads as stuck, even when you are not.

- Get a correct simple solution first, then improve it, and say its cost in plain words ("one pass over the list, plus a dict").

- Test it yourself on your example and on one edge case before you say "done".

Use only what you know from Days 1 to 4: loops, functions, list, dict and set, no libraries. Do the two Day 4 DSA problems first; they are the same level.

## LinkedIn and portfolio action

This week's action is the repo itself. A hiring manager gives a GitHub profile about a minute, so the README has to say who you are and what the repo proves before they scroll.

- Bring in the Day 5 to 13 folders first: unzip the files from the project chat into fde-journey/, or apply each patch with git am. Then the README describes a repo that matches the plan.

- Rewrite the top of README.md: one line on who you are (14 years, headless CMS, moving into FDE work), one line on what the repo is, and the day table with a Status column (done, in progress, not started). Keep it honest; "in progress" is fine.

- On GitHub, add a one-line repo description and topics (python, sql, fastapi, llm, rag), and pin the repo on your profile.

- If the Day 7 journey post is not live yet, publish it this week, after Day 4 is pushed. Post about what you built, not what you read.

## DSA practice (30 min)

No new problems. Finish the two in day04/dsa.py, which is already in your repo with the signatures, examples and hints.

- Top k words. top_words(text, k). Hint: count with a dict as on Day 3, then sort the (word, count) pairs with one key that puts higher counts first and breaks ties alphabetically. Think about what a tuple key with a minus sign does.

- Balanced brackets. is_balanced(s). Hint: a list works as a stack. Map each closing bracket to its opening one, and ask what must be true about the stack when the loop ends.

## Quiz (3 quick questions)

- A customer asks whether your CMS exporter can run every night in their CI pipeline. Name three things from Week 2 that let you say yes.

- You notice that a pull request merged yesterday added a .env file holding the customer's API key. What do you do, in order?

- Explain to a non-technical product owner, in two sentences, why the exporter sends at most 3 requests at a time even though 10 would look faster.

## What you achieved this week

Week 2 covered the habits that make code safe to hand to a customer: Git with pull requests, the shell, tests, generators and context managers, packaging with uv, and async calls with limits. Today you measured how much of that is in your hands yet, chose what to finish and what to postpone, and set up Week 3, where your CMS data moves into a real database.

Tomorrow (Day 15): relational thinking and your first SQL in PostgreSQL: tables, keys, data types, CREATE TABLE, INSERT and SELECT, on a CMS content model.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 14 review": your checklist with yes or no on each line, your recap answers, your predictions next to the real outputs, the link to your latest commit (Day 4 finished, Day 5 to 13 folders in), the new README top section, your quiz answers, and how many hours a day you can really study for the next two weeks. Then run the coding mock in the same chat.

## Recap answers

*Your answers:*
